In [3]:
import sys, subprocess, importlib.util
required = {"kagglehub":"kagglehub>=1.0.2", "imblearn":"imbalanced-learn>=0.12", "lightgbm":"lightgbm", "xgboost":"xgboost", "shap":"shap", "joblib":"joblib"}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
print("Dependencies ready")

Dependencies ready


In [4]:
import os, json, math, time, hashlib, zipfile, warnings, platform, inspect
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import softmax
from scipy.optimize import minimize_scalar
from scipy.stats import wilcoxon
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold, ParameterGrid
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix, matthews_corrcoef, cohen_kappa_score, roc_auc_score, average_precision_score, log_loss, brier_score_loss, precision_recall_curve, roc_curve)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
import joblib, kagglehub, shap
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT=Path('/kaggle/working/metabolic_revision') if Path('/kaggle/working').exists() else Path.cwd()/'metabolic_revision'
for p in ['tables','figures','weights','logs','metadata','audit','probabilities']:
    (ROOT/p).mkdir(parents=True,exist_ok=True)
SEED=42
np.random.seed(SEED)
print('Working directory:',ROOT)
print('Python:',platform.python_version())

Working directory: /kaggle/working/metabolic_revision
Python: 3.13.16


In [5]:
CFG={
    'dataset_handle':'parasharmanu/close-to-realistic-calorie-efficiency-dataset',
    'main_sample_n':50000,
    'train_fraction':0.70,
    'validation_fraction':0.15,
    'test_fraction':0.15,
    'calibration_fraction_of_training':0.20,
    'cv_folds':5,
    'outer_cv_folds':5,
    'inner_cv_folds':3,
    'final_seeds':[11,23,42,71,101],
    'sample_sizes':[50000,100000,250000,500000,1000000],
    'bootstrap_repetitions':400,
    'shap_sample_size':600,
    'use_gpu_for_xgboost':False,
    'run_conditional_nested_cv':True,
    'run_sample_size_robustness':True,
    'run_full_data_fit':True,
    'run_stratified_subgroup_audit':True,
    'run_class_specific_shap':True,
    'strict_label_rule_audit':False,
    'max_training_seconds_per_fit':None,
    'figure_dpi':450
}
assert abs(CFG['train_fraction']+CFG['validation_fraction']+CFG['test_fraction']-1)<1e-10
(ROOT/'metadata'/'configuration.json').write_text(json.dumps(CFG,indent=2))
print(json.dumps(CFG,indent=2))

{
  "dataset_handle": "parasharmanu/close-to-realistic-calorie-efficiency-dataset",
  "main_sample_n": 50000,
  "train_fraction": 0.7,
  "validation_fraction": 0.15,
  "test_fraction": 0.15,
  "calibration_fraction_of_training": 0.2,
  "cv_folds": 5,
  "outer_cv_folds": 5,
  "inner_cv_folds": 3,
  "final_seeds": [
    11,
    23,
    42,
    71,
    101
  ],
  "sample_sizes": [
    50000,
    100000,
    250000,
    500000,
    1000000
  ],
  "bootstrap_repetitions": 400,
  "shap_sample_size": 600,
  "use_gpu_for_xgboost": false,
  "run_conditional_nested_cv": true,
  "run_sample_size_robustness": true,
  "run_full_data_fit": true,
  "run_stratified_subgroup_audit": true,
  "run_class_specific_shap": true,
  "strict_label_rule_audit": false,
  "max_training_seconds_per_fit": null,
  "figure_dpi": 450
}


In [6]:
handle=CFG['dataset_handle']
try:
    dataset_dir=Path(kagglehub.dataset_download(handle))
except Exception as error:
    attached=list(Path('/kaggle/input').rglob('*.csv'))+list(Path('/kaggle/input').rglob('*.parquet')) if Path('/kaggle/input').exists() else []
    attached=[p for p in attached if any(z in str(p).lower() for z in ['calorie','efficiency','metabolic'])]
    if not attached:
        raise RuntimeError('Automatic dataset retrieval failed. Enable Kaggle Internet or attach the public dataset first. The original error was: '+str(error)) from error
    dataset_dir=attached[0].parent
files=[p for p in dataset_dir.rglob('*') if p.is_file()]
data_files=[p for p in files if p.suffix.lower() in {'.csv','.parquet','.feather'}]
if not data_files:
    raise FileNotFoundError('No tabular data file found in '+str(dataset_dir))
source=max(data_files,key=lambda p:p.stat().st_size)
dataset_sha256=hashlib.sha256()
with source.open('rb') as handle_f:
    for block in iter(lambda:handle_f.read(4*1024*1024),b''):
        dataset_sha256.update(block)
raw=pd.read_csv(source,low_memory=False) if source.suffix.lower()=='.csv' else (pd.read_parquet(source) if source.suffix.lower()=='.parquet' else pd.read_feather(source))
source_manifest={'dataset_handle':CFG['dataset_handle'],'dataset_path':str(source),'sha256':dataset_sha256.hexdigest(),'rows':len(raw),'columns':raw.columns.tolist(),'downloaded_utc':datetime.now(timezone.utc).isoformat(),'source_files':[str(p.relative_to(dataset_dir)) for p in files]}
(ROOT/'metadata'/'dataset_manifest.json').write_text(json.dumps(source_manifest,indent=2,default=str))
print('Loaded:',source,'Shape:',raw.shape)
print('Column names:',raw.columns.tolist())
print(raw.head(3).to_string(index=False))

Loaded: /kaggle/input/datasets/parasharmanu/close-to-realistic-calorie-efficiency-dataset/calorie_efficiency_dataset.csv Shape: (1000000, 15)
Column names: ['age', 'steps_per_day', 'active_minutes', 'calories_burned', 'sleep_hours', 'hydration_liters', 'bmi', 'workouts_per_week', 'muscle_mass_ratio', 'body_fat_percentage', 'heart_rate_resting', 'heart_rate_avg', 'continuous_exercise_days', 'efficiency_score', 'calorie_efficiency']
 age  steps_per_day  active_minutes  calories_burned  sleep_hours  hydration_liters   bmi  workouts_per_week  muscle_mass_ratio  body_fat_percentage  heart_rate_resting  heart_rate_avg  continuous_exercise_days  efficiency_score calorie_efficiency
  51           7853              99             1500         6.42              3.60 22.34                  3              0.321                 0.05                68.5           102.0                         1             0.603     Low Efficiency
  60           4820              78             1500         6.82    

In [7]:
import re

def norm_column(s):
    return re.sub(r'_+','_',re.sub(r'[^a-z0-9]+','_',str(s).lower())).strip('_')
original_names=list(raw.columns)
normalized=[norm_column(s) for s in original_names]
if len(set(normalized))!=len(normalized):
    raise ValueError('Column normalization produced duplicate names: '+str(normalized))
raw.columns=normalized
label_candidates=[x for x in raw.columns if re.search(r'(^|_)(efficiency|target|label|class|category)(_|$)',x) and not re.search(r'(^|_)(score|ratio|index)(_|$)',x)]
def target_quality(s):
    nonnull=s.dropna()
    vals=nonnull.astype(str).str.lower().str.strip().unique()
    lowmodhigh=all(any(re.search(r'(^|[^a-z])'+v+r'([^a-z]|$)',z) for z in vals) for v in ['low','moderate','high'])
    three=len(vals)==3
    return (100 if lowmodhigh else 0)+(15 if three else 0)+(2 if len(vals)<=8 else 0)
label_candidates=sorted(label_candidates,key=lambda x:target_quality(raw[x]),reverse=True)
if not label_candidates:
    label_candidates=sorted([c for c in raw.columns if raw[c].nunique(dropna=True)==3],key=lambda x:target_quality(raw[x]),reverse=True)
if not label_candidates:
    raise ValueError('Unable to identify a three-class target. Check dataset source and printed schema.')
TARGET=label_candidates[0]
if raw[TARGET].nunique(dropna=True)!=3:
    raise ValueError('Potential target '+TARGET+' does not have exactly three classes. Candidate columns: '+str(label_candidates))
raw_labels=raw[TARGET].astype(str).str.strip()
unique_label=list(raw_labels.unique())
def class_name(v):
    s=v.lower().strip()
    for name in ('low','moderate','high'):
        if re.search(r'(^|[^a-z])'+name+r'([^a-z]|$)',s):
            return name.title()
    return v
label_names=[class_name(x) for x in unique_label]
if len(set(label_names))!=3:
    raise ValueError('Ambiguous label normalization: '+str(dict(zip(unique_label,label_names))))
if set(label_names)=={'Low','Moderate','High'}:
    classes=['Low','Moderate','High']
else:
    classes=sorted(label_names)
lookup={k:classes.index(v) for k,v in zip(unique_label,label_names)}
y_all=raw_labels.map(lookup).astype(int)
print('Target:',TARGET,'Class mapping:',lookup,'Class counts:',y_all.value_counts().sort_index().to_dict())
print('Other candidate label columns:',label_candidates[1:])
assert len(classes)==3

Target: calorie_efficiency Class mapping: {'Low Efficiency': 0, 'Moderate': 1, 'High Efficiency': 2} Class counts: {0: 938243, 1: 34861, 2: 26896}
Other candidate label columns: []


In [8]:
script_files=[p for p in files if p.suffix.lower() in {'.py','.ipynb','.md','.txt','.json'} and p.stat().st_size<3_000_000]
rule_hits=[]
for script in script_files:
    try:
        snippet=script.read_text(encoding='utf8',errors='ignore')
        lines=snippet.splitlines()
        for i,line in enumerate(lines):
            if re.search(r'(?i)(efficiency|target|label|class)',line) and re.search(r'(?i)(if |where\(|cut\(|percentile|threshold|quantile|lambda|np\.select|choice)',line):
                rule_hits.append({'file':str(script.relative_to(dataset_dir)),'line':i+1,'text':line[:500]})
    except Exception:
        pass
pd.DataFrame(rule_hits,columns=['file','line','text']).to_csv(ROOT/'audit'/'potential_label_generation_source_lines.csv',index=False)
label_rule_verified=False
label_rule_status='NOT VERIFIED: inspect creator documentation or original data-generation code; heuristics and decision trees cannot prove label definitions'
print('Label generation source candidates:',len(rule_hits))
print('Target definition audit:',label_rule_status)
if CFG['strict_label_rule_audit'] and not label_rule_verified:
    raise RuntimeError('The label-generation formula is not independently verified. Supply authoritative generator code before manuscript claims.')
source_column_audit=pd.DataFrame({'column':raw.columns,'dtype':[str(t) for t in raw.dtypes],'missing':raw.isna().sum().values,'unique':raw.nunique(dropna=False).values})
source_column_audit.to_csv(ROOT/'audit'/'source_column_audit.csv',index=False)
print(source_column_audit.to_string(index=False))

Label generation source candidates: 0
Target definition audit: NOT VERIFIED: inspect creator documentation or original data-generation code; heuristics and decision trees cannot prove label definitions
                  column   dtype  missing  unique
                     age   int64        0      47
           steps_per_day   int64        0   14382
          active_minutes   int64        0     171
         calories_burned   int64        0       1
             sleep_hours float64        0     701
        hydration_liters float64        0     451
                     bmi float64        0    2401
       workouts_per_week   int64        0       8
       muscle_mass_ratio float64        0     401
     body_fat_percentage float64        0     451
      heart_rate_resting float64        0     380
          heart_rate_avg float64        0     735
continuous_exercise_days   int64        0       7
        efficiency_score float64        0    6618
      calorie_efficiency  object        0       

In [9]:
dataset=raw.copy()
dataset.insert(0,'source_row_id',np.arange(len(dataset),dtype=np.int64))
excluded={TARGET,'source_row_id'}
identifier_cols=[c for c in dataset if c not in excluded and (re.search(r'(^|_)(id|identifier|uuid|name|email|session_id|timestamp|date)(_|$)',c) is not None)]
label_like=[c for c in dataset if c not in excluded and c not in identifier_cols and (re.search(r'(efficiency|target|label|class|category|calorie_burn_efficiency)',c) is not None)]
feature_candidates=[c for c in dataset if c not in excluded and c not in identifier_cols and c not in label_like]
non_numeric=[c for c in feature_candidates if not pd.api.types.is_numeric_dtype(dataset[c])]
if non_numeric:
    print('Non-numeric predictors excluded pending a documented categorical treatment:',non_numeric)
feature_candidates=[c for c in feature_candidates if c not in non_numeric]
if len(feature_candidates)<5:
    raise ValueError('Insufficient numerical predictors after excluding target-like columns: '+str(feature_candidates))
for c in feature_candidates:
    dataset[c]=pd.to_numeric(dataset[c],errors='coerce')
constant=[c for c in feature_candidates if dataset[c].nunique(dropna=True)<=1]
feature_candidates=[c for c in feature_candidates if c not in constant]
if not feature_candidates:
    raise RuntimeError('All features are constant')
y_label=y_all.to_numpy()
feature_hash=pd.util.hash_pandas_object(dataset[feature_candidates],index=False).to_numpy()
conflicts=pd.DataFrame({'hash':feature_hash,'label':y_label}).groupby('hash')['label'].nunique()
conflicting_hashes=set(conflicts[conflicts>1].index)
conflict_mask=np.isin(feature_hash,list(conflicting_hashes)) if conflicting_hashes else np.zeros(len(dataset),dtype=bool)
duplicate_mask=pd.Series(feature_hash).duplicated(keep='first').to_numpy()
keep=~(conflict_mask|duplicate_mask)
audit_dedup={'initial_rows':int(len(dataset)),'duplicate_feature_rows':int(duplicate_mask.sum()),'rows_with_conflicting_feature_label':int(conflict_mask.sum()),'remaining_rows':int(keep.sum()),'constant_fields_excluded':constant,'id_fields_excluded':identifier_cols,'target_like_fields_excluded':label_like,'non_numeric_fields_excluded':non_numeric}
(ROOT/'audit'/'data_integrity.json').write_text(json.dumps(audit_dedup,indent=2))
dataset=dataset.loc[keep].reset_index(drop=True)
dataset[TARGET]=np.asarray(y_label)[keep]
if dataset[TARGET].value_counts().min()<30:
    raise RuntimeError('Too few independent minority records for 5-fold evaluation')
print('Integrity audit:',json.dumps(audit_dedup,indent=2))
print('Predictor columns:',feature_candidates)

Integrity audit: {
  "initial_rows": 1000000,
  "duplicate_feature_rows": 0,
  "rows_with_conflicting_feature_label": 0,
  "remaining_rows": 1000000,
  "constant_fields_excluded": [
    "calories_burned"
  ],
  "id_fields_excluded": [],
  "target_like_fields_excluded": [
    "efficiency_score"
  ],
  "non_numeric_fields_excluded": []
}
Predictor columns: ['age', 'steps_per_day', 'active_minutes', 'sleep_hours', 'hydration_liters', 'bmi', 'workouts_per_week', 'muscle_mass_ratio', 'body_fat_percentage', 'heart_rate_resting', 'heart_rate_avg', 'continuous_exercise_days']


In [10]:
feature_registry={
 'heart_rate_reserve_fraction':{'formula':'(heart_rate_avg - heart_rate_resting) / (220 - age - heart_rate_resting)','inputs':['heart_rate_avg','heart_rate_resting','age'],'status':'documented_original_equation'},
 'weekly_workout_volume':{'formula':'workouts_per_week * active_minutes','inputs':['workouts_per_week','active_minutes'],'status':'documented_original_equation'},
 'recovery_adequacy_index':{'formula':'(sleep_hours * hydration_liters) / (workouts_per_week + 1)','inputs':['sleep_hours','hydration_liters','workouts_per_week'],'status':'documented_original_equation'},
 'exercise_consistency_flag':{'formula':'1[continuous_exercise_days >= 5]','inputs':['continuous_exercise_days'],'status':'documented_original_equation'},
 'cardiac_load_ratio':{'formula':'heart_rate_avg / max(heart_rate_resting, 1)','inputs':['heart_rate_avg','heart_rate_resting'],'status':'NEW exploration; not the original inverse calorie/cardiac-effort ratio'},
 'sleep_workout_discordance':{'formula':'workouts_per_week / (sleep_hours + 1)','inputs':['workouts_per_week','sleep_hours'],'status':'NEW exploration; not the original overtraining-risk flag'},
 'adiposity_interaction':{'formula':'bmi * body_fat_percentage','inputs':['bmi','body_fat_percentage'],'status':'NEW exploration; replaces non-identifiable BMR proxy; not an estimate of BMR'}
}
aliases={'heart_rate_avg':['heart_rate_avg','average_heart_rate','heart_rate_average','heart_rate_exercise','heart_rate'],
'heart_rate_resting':['heart_rate_resting','resting_heart_rate','heart_rate_rest','resting_hr'],
'age':['age'], 'workouts_per_week':['workouts_per_week','workout_frequency','weekly_workouts'],
'active_minutes':['active_minutes','active_minutes_per_day','active_min'],
'sleep_hours':['sleep_hours','sleep_duration','hours_of_sleep'],
'hydration_liters':['hydration_liters','water_intake_liters','water_intake'],
'continuous_exercise_days':['continuous_exercise_days','exercise_streak','consecutive_exercise_days'],
'bmi':['bmi'], 'body_fat_percentage':['body_fat_percentage','body_fat_pct','body_fat']}
resolved={k:next((n for n in opts if n in feature_candidates),None) for k,opts in aliases.items()}
feature_registry_used={k:v for k,v in feature_registry.items() if all(resolved[s] is not None for s in v['inputs'])}
feature_registry_skipped={k:v for k,v in feature_registry.items() if k not in feature_registry_used}
(ROOT/'metadata'/'feature_definitions.json').write_text(json.dumps({'all_definitions':feature_registry,'resolved_columns':resolved,'used':list(feature_registry_used),'skipped':list(feature_registry_skipped),'unrecovered_original_formulas':['inverse cardiac-effort ratio','overtraining-risk indicator','BMR proxy']},indent=2))
print('Exact formulas implemented:',list(feature_registry_used))
print('Missing-input formulas:',list(feature_registry_skipped))

def engineer(frame,mode='full_seven'):
    z=frame[feature_candidates].copy().astype('float32')
    if mode=='raw_only':
        return z
    g=lambda key:z[resolved[key]].astype('float64')
    def safe_ratio(num,den):
        d=np.asarray(den,dtype=float)
        d=np.where(np.abs(d)<1e-5,np.nan,d)
        return pd.Series(np.asarray(num,dtype=float)/d,index=z.index,dtype=float)
    funcs={
      'heart_rate_reserve_fraction':lambda:safe_ratio(g('heart_rate_avg')-g('heart_rate_resting'),220-g('age')-g('heart_rate_resting')),
      'weekly_workout_volume':lambda:g('workouts_per_week')*g('active_minutes'),
      'recovery_adequacy_index':lambda:safe_ratio(g('sleep_hours')*g('hydration_liters'),g('workouts_per_week')+1),
      'exercise_consistency_flag':lambda:(g('continuous_exercise_days')>=5).astype('float32'),
      'cardiac_load_ratio':lambda:safe_ratio(g('heart_rate_avg'),np.maximum(g('heart_rate_resting'),1)),
      'sleep_workout_discordance':lambda:safe_ratio(g('workouts_per_week'),g('sleep_hours')+1),
      'adiposity_interaction':lambda:g('bmi')*g('body_fat_percentage')
    }
    for key in (list(feature_registry)[:4] if mode=='core_four' else list(feature_registry)):
        if key in feature_registry_used:
            z[key]=funcs[key]().astype('float32')
    return z.replace([np.inf,-np.inf],np.nan)
print('Feature engineering modes ready')

Exact formulas implemented: ['heart_rate_reserve_fraction', 'weekly_workout_volume', 'recovery_adequacy_index', 'exercise_consistency_flag', 'cardiac_load_ratio', 'sleep_workout_discordance', 'adiposity_interaction']
Missing-input formulas: []
Feature engineering modes ready


In [11]:
strata=dataset[TARGET].to_numpy()
all_indices=np.arange(len(dataset))
main_n=min(CFG['main_sample_n'],len(dataset))
if main_n<len(dataset):
    sampled,unused=train_test_split(all_indices,train_size=main_n,stratify=strata,random_state=SEED)
else:
    sampled=all_indices
main=dataset.iloc[np.sort(sampled)].copy().reset_index(drop=True)
from sklearn.model_selection import GroupShuffleSplit
valid_group_cols=[c for c in ['patient_id','participant_id','subject_id','person_id','user_id'] if c in main.columns and 1<main[c].nunique()<len(main)]
GROUP_COL=valid_group_cols[0] if valid_group_cols else None

def group_or_stratified_split(frame,test_fraction,seed):
    if GROUP_COL:
        splitter=GroupShuffleSplit(n_splits=1,test_size=test_fraction,random_state=seed)
        a,b=next(splitter.split(frame,frame[TARGET],groups=frame[GROUP_COL]))
        left,right=frame.iloc[a].copy(),frame.iloc[b].copy()
        if left[TARGET].nunique()<len(classes) or right[TARGET].nunique()<len(classes):
            raise RuntimeError('Group-disjoint splitting left a class absent. Original patient IDs need an adjusted group-aware plan; do not silently revert to row splitting.')
        return left,right
    return train_test_split(frame,test_size=test_fraction,stratify=frame[TARGET],random_state=seed)

train_block,test_block=group_or_stratified_split(main,CFG['test_fraction'],SEED)
train_block,val_block=group_or_stratified_split(train_block,CFG['validation_fraction']/(1-CFG['test_fraction']),SEED)
train_block=train_block.reset_index(drop=True)
val_block=val_block.reset_index(drop=True)
test_block=test_block.reset_index(drop=True)
fit_block,cal_block=group_or_stratified_split(train_block,CFG['calibration_fraction_of_training'],SEED)
fit_block=fit_block.reset_index(drop=True)
cal_block=cal_block.reset_index(drop=True)

def cv_split(frame,n_splits,seed):
    if GROUP_COL:
        kf=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
        return kf.split(frame,frame[TARGET].to_numpy(),groups=frame[GROUP_COL].to_numpy())
    kf=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    return kf.split(frame,frame[TARGET].to_numpy())
for name,frame in [('fit',fit_block),('calibration',cal_block),('validation',val_block),('test_locked',test_block)]:
    frame[['source_row_id',TARGET]].to_csv(ROOT/'metadata'/f'{name}_source_identifiers.csv',index=False)
    print(name,len(frame),frame[TARGET].value_counts().sort_index().to_dict())
assert not (set(train_block.source_row_id)&set(val_block.source_row_id))
assert not (set(train_block.source_row_id)&set(test_block.source_row_id))
assert not (set(val_block.source_row_id)&set(test_block.source_row_id))
patient_group_columns=[x for x in ['patient_id','participant_id','subject_id','person_id','user_id','device_id','site_id'] if x in dataset.columns]
if patient_group_columns:
    print('Potential grouping keys exist:',patient_group_columns,'; verify their meaning before claiming patient/site-disjoint evaluation')
else:
    print('No verifiable patient/site IDs; five-fold CV will be row-stratified, not patient grouped')
plt.figure(figsize=(8,4))
main[TARGET].map(dict(enumerate(classes))).value_counts().reindex(classes).plot.bar()
plt.ylabel('Observations');plt.title('Main dataset class distribution');plt.tight_layout()
plt.savefig(ROOT/'figures'/'class_distribution.png',dpi=CFG['figure_dpi']);plt.close()
print('Locked test hashes:',hashlib.sha256(test_block.source_row_id.to_numpy().tobytes()).hexdigest())

fit 27999 {0: 26269, 1: 977, 2: 753}
calibration 7000 {0: 6568, 1: 244, 2: 188}
validation 7501 {0: 7038, 1: 261, 2: 202}
test_locked 7500 {0: 7037, 1: 261, 2: 202}
No verifiable patient/site IDs; five-fold CV will be row-stratified, not patient grouped
Locked test hashes: b667b9630c429d2b22b54a959605d10339e4a66fd5e3db5cefbbcb8c714cb476


In [12]:
def fit_transform_preprocess(xtrain,xother=None):
    xtrain=xtrain.replace([np.inf,-np.inf],np.nan)
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    train=np.asarray(imputer.fit_transform(xtrain),dtype=np.float32)
    lower=np.quantile(train,0.01,axis=0).astype('float32')
    upper=np.quantile(train,0.99,axis=0).astype('float32')
    train=np.clip(train,lower,upper)
    scaler=RobustScaler()
    train=scaler.fit_transform(train).astype('float32')
    obj={'imputer':imputer,'lower':lower,'upper':upper,'scaler':scaler,'columns':list(xtrain.columns)}
    if xother is None:
        return train,obj
    return train,transform_preprocess(obj,xother),obj

def transform_preprocess(obj,x):
    missing=set(obj['columns'])-set(x.columns)
    if missing:
        raise ValueError('Missing expected engineered features: '+str(missing))
    a=np.asarray(obj['imputer'].transform(x[obj['columns']].replace([np.inf,-np.inf],np.nan)),dtype=np.float32)
    a=np.clip(a,obj['lower'],obj['upper'])
    return np.asarray(obj['scaler'].transform(a),dtype=np.float32)

def model_instance(name,params,seed):
    if name=='RandomForest':
        return RandomForestClassifier(random_state=seed,n_jobs=-1,class_weight=None,**params)
    if name=='XGBoost':
        device='cuda' if CFG['use_gpu_for_xgboost'] else 'cpu'
        return XGBClassifier(objective='multi:softprob',num_class=len(classes),random_state=seed,tree_method='hist',device=device,n_jobs=2,verbosity=0,eval_metric='mlogloss',**params)
    if name=='LightGBM':
        return LGBMClassifier(objective='multiclass',num_class=len(classes),random_state=seed,n_jobs=2,verbosity=-1,**params)
    raise KeyError(name)

PARAMS={
'RandomForest':[{'n_estimators':250,'max_depth':None,'min_samples_leaf':1,'max_features':'sqrt'},
                {'n_estimators':350,'max_depth':14,'min_samples_leaf':2,'max_features':'sqrt'},
                {'n_estimators':300,'max_depth':9,'min_samples_leaf':4,'max_features':0.75}],
'XGBoost':[{'n_estimators':320,'max_depth':4,'learning_rate':0.05,'subsample':0.85,'colsample_bytree':0.85},
           {'n_estimators':500,'max_depth':6,'learning_rate':0.045,'subsample':0.9,'colsample_bytree':0.9},
           {'n_estimators':450,'max_depth':3,'learning_rate':0.075,'subsample':0.8,'colsample_bytree':0.9}],
'LightGBM':[{'n_estimators':300,'num_leaves':31,'learning_rate':0.05,'min_child_samples':35,'subsample':0.85,'colsample_bytree':0.85},
            {'n_estimators':450,'num_leaves':63,'learning_rate':0.04,'min_child_samples':50,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':380,'num_leaves':15,'learning_rate':0.08,'min_child_samples':70,'subsample':0.9,'colsample_bytree':0.85}]
}
BALANCING=['none','class_weights','smote','smote_plus_weights']

def train_pipeline(raw_train,labels,name,params,strategy,seed,feature_mode='full_seven',excluded_features=()):
    x=engineer(raw_train,feature_mode).drop(columns=list(excluded_features),errors='ignore')
    a,pre=fit_transform_preprocess(x)
    yi=np.asarray(labels,dtype=int)
    original_counts=np.bincount(yi,minlength=len(classes))
    original_weights=len(yi)/(len(classes)*np.maximum(original_counts,1))
    if strategy in ['smote','smote_plus_weights']:
        min_count=int(np.bincount(yi).min())
        if min_count>1:
            sampler=SMOTE(random_state=seed,k_neighbors=min(5,min_count-1))
            a,yi=sampler.fit_resample(a,yi)
        else:
            raise ValueError('Not enough minority observations for SMOTE')
    weights=original_weights[yi] if strategy in ['class_weights','smote_plus_weights'] else None
    estimator=model_instance(name,params,seed)
    estimator.fit(a,yi,sample_weight=weights)
    return {'preprocessor':pre,'estimator':estimator,'model':name,'params':params,'balancing':strategy,'seed':seed,'feature_mode':feature_mode,'excluded_features':tuple(excluded_features)}

def pipeline_proba(bundle,data):
    x=engineer(data,bundle['feature_mode']).drop(columns=list(bundle['excluded_features']),errors='ignore')
    return np.asarray(bundle['estimator'].predict_proba(transform_preprocess(bundle['preprocessor'],x)),dtype=np.float64)

def numeric_summary(y,p,threshold=0.5):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    predicted=p.argmax(axis=1)
    out={'accuracy':accuracy_score(y,predicted),'balanced_accuracy':balanced_accuracy_score(y,predicted),'macro_f1':f1_score(y,predicted,average='macro',zero_division=0),
         'weighted_f1':f1_score(y,predicted,average='weighted',zero_division=0),'mcc':matthews_corrcoef(y,predicted),'kappa':cohen_kappa_score(y,predicted),
         'nll':log_loss(y,np.clip(p,1e-8,1),labels=np.arange(len(classes))),
         'multiclass_brier':float(np.mean(np.sum((p-np.eye(len(classes))[y])**2,axis=1)))}
    ce=np.max(p,axis=1)
    correct=(predicted==y).astype(float)
    out['ece_15bins']=float(sum(np.abs(correct[(ce>=lo)&(ce<(hi if hi<1 else 1.00001))].mean()-ce[(ce>=lo)&(ce<(hi if hi<1 else 1.00001))].mean())*np.sum((ce>=lo)&(ce<(hi if hi<1 else 1.00001)))/len(y) for lo,hi in zip(np.linspace(0,1,16)[:-1],np.linspace(0,1,16)[1:]) if np.any((ce>=lo)&(ce<(hi if hi<1 else 1.00001)))))
    out['macro_ovr_ap']=float(np.mean([average_precision_score((y==c).astype(int),p[:,c]) for c in range(len(classes)) if np.any(y==c)]))
    try:out['macro_ovr_roc_auc']=roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    except ValueError:out['macro_ovr_roc_auc']=np.nan
    for j,c in enumerate(classes):
        binary=(y==j).astype(int)
        out[f'{c}_prevalence']=binary.mean()
        out[f'{c}_recall']=float(np.mean(predicted[y==j]==j)) if np.any(y==j) else np.nan
        out[f'{c}_precision']=float(np.sum((predicted==j)&(y==j))/max(np.sum(predicted==j),1))
        out[f'{c}_f1']=f1_score(binary,(predicted==j).astype(int),zero_division=0)
        out[f'{c}_ap']=average_precision_score(binary,p[:,j]) if 0<binary.sum()<len(binary) else np.nan
    return out
print('Models, sampler, preprocessing, and metrics ready')

Models, sampler, preprocessing, and metrics ready


In [13]:
CV_X=fit_block
CV_y=fit_block[TARGET].to_numpy(dtype=int)
cv=None
cv_rows=[]
start=time.time()
for name in PARAMS:
    for strategy in BALANCING:
        for p_idx,param in enumerate(PARAMS[name]):
            for fold,(it,iv) in enumerate(cv_split(CV_X,CFG['cv_folds'],SEED),1):
                fit_data=CV_X.iloc[it]
                validation_data=CV_X.iloc[iv]
                trained=train_pipeline(fit_data,fit_data[TARGET].to_numpy(),name,param,strategy,seed=SEED+fold)
                q=pipeline_proba(trained,validation_data)
                stats=numeric_summary(validation_data[TARGET].to_numpy(),q)
                cv_rows.append({'model':name,'balance':strategy,'hyperparam_id':p_idx,'fold':fold,**stats})
            print(name,strategy,'setting',p_idx,'completed in',round((time.time()-start)/60,1),'minutes')
cv_results=pd.DataFrame(cv_rows)
cv_results.to_csv(ROOT/'tables'/'fivefold_comprehensive_cv.csv',index=False)
cv_aggregate=cv_results.groupby(['model','balance','hyperparam_id'],as_index=False).agg(macro_f1_mean=('macro_f1','mean'),macro_f1_std=('macro_f1','std'),balanced_accuracy_mean=('balanced_accuracy','mean'),high_recall_mean=(f'{"High" if "High" in classes else classes[-1]}_recall','mean'),macro_ap_mean=('macro_ovr_ap','mean'),brier_mean=('multiclass_brier','mean'))
cv_aggregate=cv_aggregate.sort_values(['macro_f1_mean','macro_ap_mean'],ascending=False)
cv_aggregate.to_csv(ROOT/'tables'/'fivefold_cv_ranked.csv',index=False)
print(cv_aggregate.head(15).to_string(index=False))

RandomForest none setting 0 completed in 0.9 minutes
RandomForest none setting 1 completed in 1.7 minutes
RandomForest none setting 2 completed in 3.1 minutes
RandomForest class_weights setting 0 completed in 3.9 minutes
RandomForest class_weights setting 1 completed in 4.6 minutes
RandomForest class_weights setting 2 completed in 6.1 minutes
RandomForest smote setting 0 completed in 9.4 minutes
RandomForest smote setting 1 completed in 12.6 minutes
RandomForest smote setting 2 completed in 18.8 minutes
RandomForest smote_plus_weights setting 0 completed in 22.4 minutes
RandomForest smote_plus_weights setting 1 completed in 25.6 minutes
RandomForest smote_plus_weights setting 2 completed in 31.8 minutes


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


XGBoost none setting 0 completed in 32.0 minutes


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


XGBoost none setting 1 completed in 32.3 minutes


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


... (39 outputs left out)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM smote_plus_weights setting 1 completed in 39.9 minutes


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM smote_plus_weights setting 2 completed in 40.2 minutes
       model            balance  hyperparam_id  macro_f1_mean  macro_f1_std  balanced_accuracy_mean  high_recall_mean  macro_ap_mean  brier_mean
RandomForest      class_weights              2       0.443826      0.005085                0.412770          0.031903       0.415211    0.467365
    LightGBM      class_weights              1       0.442307      0.008741                0.413066          0.002658       0.413292    0.120703
RandomForest      class_weights              1       0.440592      0.009192                0.407737          0.000000       0.415168    0.260074
RandomForest               none              2       0.439263      0.005472                0.405638          0.000000       0.415011    0.105498
    LightGBM              smote              0       0.439251      0.008042                0.409024          0.000000       0.411217    0.125386
    LightGBM              smote              1       0.438963     

In [14]:
winners=[]
for name in PARAMS:
    best=cv_aggregate[cv_aggregate.model==name].iloc[0]
    setting=PARAMS[name][int(best.hyperparam_id)]
    strategy=best.balance
    bundle=train_pipeline(fit_block,fit_block[TARGET].to_numpy(),name,setting,strategy,seed=SEED)
    pp=pipeline_proba(bundle,val_block)
    stats=numeric_summary(val_block[TARGET].to_numpy(),pp)
    winners.append({'model':name,'balance':strategy,'hyperparam_id':int(best.hyperparam_id),'cv_macro_f1':best.macro_f1_mean,**stats})
    joblib.dump(bundle,ROOT/'weights'/f'validation_{name}.joblib',compress=3)
    pd.DataFrame(pp,columns=[f'prob_{c}' for c in classes]).assign(source_row_id=val_block.source_row_id.to_numpy(),truth=val_block[TARGET].to_numpy()).to_csv(ROOT/'probabilities'/f'validation_{name}.csv',index=False)
validation_leaders=pd.DataFrame(winners).sort_values(['macro_f1','macro_ovr_ap'],ascending=False)
validation_leaders.to_csv(ROOT/'tables'/'validation_model_comparison.csv',index=False)
selection=validation_leaders.iloc[0]
BEST_NAME=selection.model
BEST_BALANCE=selection.balance
BEST_PARAMS=PARAMS[BEST_NAME][int(selection.hyperparam_id)]
print(validation_leaders[['model','balance','cv_macro_f1','macro_f1','balanced_accuracy','macro_ovr_ap','multiclass_brier']].to_string(index=False))
print('Chosen by validation macro-F1 with macro-AP secondary:',BEST_NAME,BEST_BALANCE,BEST_PARAMS)
(ROOT/'metadata'/'model_selection.json').write_text(json.dumps({'selected_model':BEST_NAME,'selected_balance':BEST_BALANCE,'selected_parameters':BEST_PARAMS,'selection_data':'cross-validation on fit portion and 15 percent validation; test untouched','primary_metric':'macro_f1','secondary_metric':'macro_ovr_ap'},indent=2,default=str))

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


       model       balance  cv_macro_f1  macro_f1  balanced_accuracy  macro_ovr_ap  multiclass_brier
RandomForest class_weights     0.443826  0.436325           0.410066      0.397168          0.493048
    LightGBM class_weights     0.442307  0.430920           0.404461      0.405033          0.133767
     XGBoost          none     0.438055  0.430443           0.399366      0.405286          0.107259
Chosen by validation macro-F1 with macro-AP secondary: RandomForest class_weights {'n_estimators': 300, 'max_depth': 9, 'min_samples_leaf': 4, 'max_features': 0.75}


373

In [15]:
def calibrate_temperature(proba,labels):
    logits=np.log(np.clip(proba,1e-10,1.0))
    y=np.asarray(labels,dtype=int)
    objective=lambda t:log_loss(y,softmax(logits/np.exp(t),axis=1),labels=np.arange(len(classes)))
    sol=minimize_scalar(objective,bounds=(-2.8,2.8),method='bounded',options={'xatol':1e-5})
    return float(np.exp(sol.x))

def temperature_apply(p,T):
    return softmax(np.log(np.clip(p,1e-10,1.0))/T,axis=1)

calibration_candidates=[]
for item in validation_leaders.to_dict('records'):
    name=item['model']
    params=PARAMS[name][int(item['hyperparam_id'])]
    trained=train_pipeline(fit_block,fit_block[TARGET].to_numpy(),name,params,item['balance'],seed=SEED)
    pc=pipeline_proba(trained,cal_block)
    t=calibrate_temperature(pc,cal_block[TARGET].to_numpy())
    pv=pipeline_proba(trained,val_block)
    for mode in ['raw','temperature']:
        q=pv if mode=='raw' else temperature_apply(pv,t)
        metrics=numeric_summary(val_block[TARGET].to_numpy(),q)
        calibration_candidates.append({'model':name,'balance':item['balance'],'calibration':mode,'temperature':t if mode=='temperature' else 1.0,**metrics})
    joblib.dump(trained,ROOT/'weights'/f'calibration_source_{name}.joblib',compress=3)
calibration_table=pd.DataFrame(calibration_candidates)
calibration_table.to_csv(ROOT/'tables'/'calibration_selection_validation.csv',index=False)
chosen_model_cal=calibration_table[calibration_table.model==BEST_NAME].sort_values(['multiclass_brier','ece_15bins']).iloc[0]
CAL_METHOD=chosen_model_cal.calibration
TEMPERATURE=float(chosen_model_cal.temperature)
print(calibration_table[['model','calibration','temperature','macro_f1','multiclass_brier','ece_15bins','nll']].to_string(index=False))
print('Selected calibration:',CAL_METHOD,'T=',TEMPERATURE)
(ROOT/'metadata'/'calibration_selection.json').write_text(json.dumps({'method':CAL_METHOD,'temperature':TEMPERATURE,'method_selected_using':'validation Brier; fitted on separate training calibration slice; not on test'},indent=2))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


       model calibration  temperature  macro_f1  multiclass_brier  ece_15bins      nll
RandomForest         raw     1.000000  0.436325          0.493048    0.456845 0.848801
RandomForest temperature     0.138275  0.436325          0.188195    0.088567 0.449050
    LightGBM         raw     1.000000  0.430920          0.133767    0.074882 0.316614
    LightGBM temperature     0.824382  0.430920          0.125002    0.052832 0.310075
     XGBoost         raw     1.000000  0.430443          0.107259    0.011781 0.256876
     XGBoost temperature     1.013876  0.430443          0.107298    0.010755 0.256728
Selected calibration: temperature T= 0.13827541880613017


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


174

In [16]:
ablation_modes=[('raw_only',()),('core_four',()),('full_seven',())]
suspected=[c for c in ['continuous_exercise_days','age','workouts_per_week','steps_per_day'] if c in feature_candidates]
if suspected:
    ablation_modes.append(('raw_only',tuple(suspected)))
    ablation_modes.append(('full_seven',tuple(suspected)))
proxied=[c for c in feature_candidates if any(t in c for t in ['efficiency','score','calorie','bmr','estimated'])]
if proxied:
    ablation_modes.append(('full_seven',tuple(proxied)))
ablations=[]
for mode,excluded in ablation_modes:
    names=set(engineer(fit_block,mode).columns)-set(excluded)
    if len(names)<2:continue
    b=train_pipeline(fit_block,fit_block[TARGET].to_numpy(),BEST_NAME,BEST_PARAMS,BEST_BALANCE,SEED,feature_mode=mode,excluded_features=excluded)
    q=pipeline_proba(b,val_block)
    s=numeric_summary(val_block[TARGET].to_numpy(),q)
    ablations.append({'feature_mode':mode,'excluded_features':','.join(excluded) or 'none','n_predictors':len(names),**s})
    joblib.dump(b,ROOT/'weights'/f'ablation_{len(ablations):02d}.joblib',compress=3)
ablation_df=pd.DataFrame(ablations).sort_values('macro_f1',ascending=False)
ablation_df.to_csv(ROOT/'tables'/'feature_ablation_validation.csv',index=False)
print(ablation_df[['feature_mode','excluded_features','n_predictors','macro_f1','macro_ovr_ap',f'{"High" if "High" in classes else classes[-1]}_recall']].to_string(index=False))
print("Removing suspected features is a stress test, NOT proof of the creator's true target rule")

feature_mode                                            excluded_features  n_predictors  macro_f1  macro_ovr_ap  High_recall
   core_four                                                         none            16  0.441647      0.399385     0.054455
  full_seven                                                         none            19  0.436325      0.397168     0.059406
    raw_only                                                         none            12  0.432843      0.401601     0.054455
  full_seven continuous_exercise_days,age,workouts_per_week,steps_per_day            15  0.401116      0.364265     0.054455
    raw_only continuous_exercise_days,age,workouts_per_week,steps_per_day             8  0.400175      0.366613     0.089109
Removing suspected features is a stress test, NOT proof of the creator's true target rule


In [17]:
fit_xy=engineer(fit_block,'raw_only')
fit_imp=SimpleImputer(strategy='median',keep_empty_features=True)
fit_array=fit_imp.fit_transform(fit_xy)
surrogate=DecisionTreeClassifier(max_depth=4,min_samples_leaf=max(40,int(len(fit_block)*0.005)),random_state=SEED)
surrogate.fit(fit_array,fit_block[TARGET].to_numpy())
surrogate_text=export_text(surrogate,feature_names=list(fit_xy.columns),max_depth=4)
(ROOT/'audit'/'surrogate_rule_tree_not_actual_generator.txt').write_text(surrogate_text)
rule_val_score=f1_score(val_block[TARGET],surrogate.predict(fit_imp.transform(engineer(val_block,'raw_only'))),average='macro')
rule_forest=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=10,n_jobs=-1,random_state=SEED)
rule_forest.fit(fit_array,fit_block[TARGET])
rule_val_prob=rule_forest.predict_proba(fit_imp.transform(engineer(val_block,'raw_only')))
rule_approx={'surrogate_tree_macro_f1_validation':float(rule_val_score),'surrogate_forest_macro_f1_validation':float(numeric_summary(val_block[TARGET],rule_val_prob)['macro_f1']),'meaning':'predictive recoverability from original raw input, not actual target generation formula'}
(ROOT/'audit'/'target_rule_recoverability.json').write_text(json.dumps(rule_approx,indent=2))
print('Target-rule recoverability diagnostic:',rule_approx)
print(surrogate_text[:3000])

Target-rule recoverability diagnostic: {'surrogate_tree_macro_f1_validation': 0.41918734078918063, 'surrogate_forest_macro_f1_validation': 0.4251588116702803, 'meaning': 'predictive recoverability from original raw input, not actual target generation formula'}
|--- steps_per_day <= 1176.50
|   |--- sleep_hours <= 6.29
|   |   |--- class: 1
|   |--- sleep_hours >  6.29
|   |   |--- class: 1
|--- steps_per_day >  1176.50
|   |--- steps_per_day <= 1573.00
|   |   |--- class: 0
|   |--- steps_per_day >  1573.00
|   |   |--- sleep_hours <= 8.01
|   |   |   |--- heart_rate_resting <= 68.65
|   |   |   |   |--- class: 0
|   |   |   |--- heart_rate_resting >  68.65
|   |   |   |   |--- class: 0
|   |   |--- sleep_hours >  8.01
|   |   |   |--- age <= 53.50
|   |   |   |   |--- class: 0
|   |   |   |--- age >  53.50
|   |   |   |   |--- class: 0



In [18]:
nested_rows=[]
if CFG['run_conditional_nested_cv']:
    population=pd.concat([fit_block,cal_block],ignore_index=True)
    nested_y=population[TARGET].to_numpy()
    outer=None
    for outer_n,(iit,iiv) in enumerate(cv_split(population,CFG['outer_cv_folds'],377),1):
        inner_frame=population.iloc[iit]
        outer_frame=population.iloc[iiv]
        inner=None
        candidates=[]
        for i,params in enumerate(PARAMS[BEST_NAME]):
            values=[]
            for fi,fv in cv_split(inner_frame,CFG['inner_cv_folds'],377+outer_n):
                dtr=inner_frame.iloc[fi]
                dva=inner_frame.iloc[fv]
                candidate=train_pipeline(dtr,dtr[TARGET].to_numpy(),BEST_NAME,params,BEST_BALANCE,SEED+outer_n+i)
                values.append(numeric_summary(dva[TARGET],pipeline_proba(candidate,dva))['macro_f1'])
            candidates.append(float(np.mean(values)))
        ix=int(np.argmax(candidates))
        trained=train_pipeline(inner_frame,inner_frame[TARGET].to_numpy(),BEST_NAME,PARAMS[BEST_NAME][ix],BEST_BALANCE,SEED+outer_n)
        scores=numeric_summary(outer_frame[TARGET],pipeline_proba(trained,outer_frame))
        nested_rows.append({'outer_fold':outer_n,'inner_choice':ix,'inner_macro_f1':max(candidates),**scores})
        print('Conditional nested fold',outer_n,'macro-F1',round(scores['macro_f1'],4))
    pd.DataFrame(nested_rows).to_csv(ROOT/'tables'/'conditional_nested_five_by_three_cv.csv',index=False)
    print('This is conditional nested tuning for a family and balancing strategy selected previously. It is not fully unbiased across every family.')

Conditional nested fold 1 macro-F1 0.443
Conditional nested fold 2 macro-F1 0.4492
Conditional nested fold 3 macro-F1 0.446
Conditional nested fold 4 macro-F1 0.4487
Conditional nested fold 5 macro-F1 0.4474
This is conditional nested tuning for a family and balancing strategy selected previously. It is not fully unbiased across every family.


In [19]:
extra_pool=dataset.loc[~dataset.source_row_id.isin(set(val_block.source_row_id)|set(test_block.source_row_id))].copy()
scaling_rows=[]
if CFG['run_sample_size_robustness']:
    for n in CFG['sample_sizes']:
        if n>=1000000 and not CFG['run_full_data_fit']:
            continue
        count=min(int(n),len(extra_pool))
        if count<500:continue
        indices=np.arange(len(extra_pool))
        if count<len(extra_pool):
            it,_=train_test_split(indices,train_size=count,stratify=extra_pool[TARGET].to_numpy(),random_state=1000+n)
            selected=extra_pool.iloc[it]
        else:
            selected=extra_pool
        to_fit=selected.copy()
        before=time.perf_counter()
        trained=train_pipeline(to_fit,to_fit[TARGET].to_numpy(),BEST_NAME,BEST_PARAMS,BEST_BALANCE,seed=SEED)
        duration=time.perf_counter()-before
        scores=numeric_summary(val_block[TARGET],pipeline_proba(trained,val_block))
        scaling_rows.append({'training_rows':len(to_fit),'requested_size':n,'fit_seconds':duration,**scores})
        print('Training size:',len(to_fit),'validation macro F1:',round(scores['macro_f1'],4),'elapsed seconds:',round(duration,1))
    pd.DataFrame(scaling_rows).to_csv(ROOT/'tables'/'sample_size_robustness_validation.csv',index=False)
    scale_df=pd.DataFrame(scaling_rows).sort_values('training_rows')
    plt.figure(figsize=(8,5))
    plt.plot(scale_df.training_rows,scale_df.macro_f1,'o-',label='Macro-F1')
    plt.plot(scale_df.training_rows,scale_df.macro_ovr_ap,'s-',label='Macro PR-AUC')
    plt.xlabel('Number of original training records');plt.ylabel('Validation metric');plt.legend();plt.tight_layout()
    plt.savefig(ROOT/'figures'/'sample_size_validation_curves.png',dpi=CFG['figure_dpi']);plt.close()
print('Scaling analysis uses validation data exclusively, not the locked test')

Training size: 50000 validation macro F1: 0.4343 elapsed seconds: 36.4
Training size: 100000 validation macro F1: 0.4221 elapsed seconds: 70.5
Training size: 250000 validation macro F1: 0.4041 elapsed seconds: 181.0
Training size: 500000 validation macro F1: 0.421 elapsed seconds: 428.0
Training size: 984999 validation macro F1: 0.4281 elapsed seconds: 788.2
Scaling analysis uses validation data exclusively, not the locked test


In [20]:
selected_bundle=joblib.load(ROOT/'weights'/f'calibration_source_{BEST_NAME}.joblib')
val_y=val_block[TARGET].to_numpy(dtype=int)
val_raw=pipeline_proba(selected_bundle,val_block)
val_proba=temperature_apply(val_raw,TEMPERATURE) if CAL_METHOD=='temperature' else val_raw

def stress(data,kind,seed):
    a=data.copy()
    rng=np.random.default_rng(seed)
    numeric=[c for c in feature_candidates if c in a.columns]
    if kind=='noise_05':
        for col in numeric:
            sd=np.nanstd(a[col].to_numpy(dtype=float))
            a[col]=a[col].to_numpy(dtype=float)+rng.normal(0,0.05*sd,len(a))
    elif kind=='noise_15':
        for col in numeric:
            sd=np.nanstd(a[col].to_numpy(dtype=float))
            a[col]=a[col].to_numpy(dtype=float)+rng.normal(0,0.15*sd,len(a))
    elif kind=='missing_10':
        for col in numeric:
            mask=rng.uniform(size=len(a))<0.10
            a.loc[mask,col]=np.nan
    elif kind=='missing_20':
        for col in numeric:
            mask=rng.uniform(size=len(a))<0.20
            a.loc[mask,col]=np.nan
    elif kind=='scale_shift_10':
        for col in numeric:
            a[col]=a[col]*1.10
    else:
        raise ValueError(kind)
    return a
robust_rows=[{'scenario':'original',**numeric_summary(val_y,val_proba)}]
for i,kind in enumerate(['noise_05','noise_15','missing_10','missing_20','scale_shift_10']):
    altered=stress(val_block,kind,SEED+i)
    q=pipeline_proba(selected_bundle,altered)
    if CAL_METHOD=='temperature':q=temperature_apply(q,TEMPERATURE)
    robust_rows.append({'scenario':kind,**numeric_summary(val_y,q)})
pd.DataFrame(robust_rows).to_csv(ROOT/'tables'/'synthetic_corruption_robustness_validation.csv',index=False)
print(pd.DataFrame(robust_rows)[['scenario','macro_f1','macro_ovr_ap','multiclass_brier']].to_string(index=False))
print('Labels are kept fixed for controlled input perturbations; these are artificial sensitivity studies, not demonstrated realistic sensor noise or external validation')

      scenario  macro_f1  macro_ovr_ap  multiclass_brier
      original  0.436325      0.397574          0.188195
      noise_05  0.436390      0.392829          0.188253
      noise_15  0.430808      0.387486          0.187695
    missing_10  0.424539      0.384329          0.203075
    missing_20  0.408038      0.373561          0.217402
scale_shift_10  0.433577      0.394863          0.128141
Labels are kept fixed for controlled input perturbations; these are artificial sensitivity studies, not demonstrated realistic sensor noise or external validation


In [21]:
subgroup_rows=[]
if CFG['run_stratified_subgroup_audit']:
    for key in ['age','bmi','body_fat_percentage','continuous_exercise_days','sleep_hours']:
        col=resolved.get(key,key) if key in resolved else key
        if col not in val_block:continue
        v=pd.to_numeric(val_block[col],errors='coerce')
        try:
            group=pd.qcut(v,q=4,duplicates='drop')
        except ValueError:
            continue
        for group_name,subframe in val_block.groupby(group,observed=True):
            ix=subframe.index.to_numpy()
            if len(ix)<50 or len(np.unique(val_y[ix]))<2:continue
            summary=numeric_summary(val_y[ix],val_proba[ix])
            subgroup_rows.append({'stratification':key,'group':str(group_name),'n':len(ix),**summary})
    pd.DataFrame(subgroup_rows).to_csv(ROOT/'tables'/'synthetic_subgroup_performance_validation.csv',index=False)
    print(pd.DataFrame(subgroup_rows)[['stratification','group','n','macro_f1','macro_ovr_ap']].head(25).to_string(index=False))
print('Subgroup slices are within the original synthetic distribution, not independent domains')

          stratification           group    n  macro_f1  macro_ovr_ap
                     age  (17.999, 29.0] 1952  0.418832      0.388902
                     age    (29.0, 41.0] 1843  0.452997      0.402548
                     age    (41.0, 53.0] 1910  0.463130      0.422353
                     age    (53.0, 64.0] 1796  0.410730      0.391339
                     bmi (15.999, 20.06] 1877  0.328626      0.332259
                     bmi  (20.06, 23.91] 1876  0.338881      0.339415
                     bmi  (23.91, 27.91] 1878  0.323736      0.332937
                     bmi   (27.91, 40.0] 1870  0.521613      0.490701
     body_fat_percentage  (0.049, 0.181] 1879  0.442196      0.395574
     body_fat_percentage   (0.181, 0.25] 1884  0.436717      0.390468
     body_fat_percentage    (0.25, 0.32] 1888  0.432503      0.406609
     body_fat_percentage     (0.32, 0.5] 1850  0.434934      0.408169
continuous_exercise_days    (0.999, 2.0] 4798  0.417511      0.390113
continuous_exercise_

In [22]:
bias_grid=[1.0,1.5,2.0,3.0,5.0,8.0]
rare_classes=[j for j in range(len(classes)) if j!=int(np.bincount(val_y).argmax())]
raw_val_predictions=val_proba.argmax(axis=1)
class_bias_search=[]
for w1 in bias_grid:
    for w2 in bias_grid:
        weights=np.ones(len(classes),dtype=float)
        for j,w in zip(rare_classes[:2],[w1,w2]):weights[j]=w
        biased=(val_proba*weights[None,:]).argmax(axis=1)
        precision,recall,f1,support=precision_recall_fscore_support(val_y,biased,labels=np.arange(len(classes)),zero_division=0)
        class_bias_search.append({'moderate_bias':w1,'high_bias':w2,'macro_f1':float(f1.mean()),'balanced_accuracy':float(recall.mean()),'rare_macro_recall':float(recall[rare_classes].mean()),'predicted_minority_fraction':float(np.mean(np.isin(biased,rare_classes))),**{f'{classes[j]}_precision':float(precision[j]) for j in range(len(classes))},**{f'{classes[j]}_recall':float(recall[j]) for j in range(len(classes))}})
bias_df=pd.DataFrame(class_bias_search).sort_values(['macro_f1','rare_macro_recall'],ascending=False)
bias_df.to_csv(ROOT/'tables'/'decision_cost_sensitivity_validation.csv',index=False)
optimum=bias_df.iloc[0]
DECISION_WEIGHTS=np.ones(len(classes),dtype=float)
for j,w in zip(rare_classes[:2],[float(optimum.moderate_bias),float(optimum.high_bias)]):DECISION_WEIGHTS[j]=w
bias_selection={'weights':DECISION_WEIGHTS.tolist(),'selected_on':'validation macro-F1 only','note':'Fixed multiplicative probability weights define an alternative decision rule; raw probabilities remain the calibrated probabilities for PR AUC and Brier','top_validation_macro_f1':float(optimum.macro_f1),'unadjusted_validation_macro_f1':float(f1_score(val_y,raw_val_predictions,average='macro',zero_division=0))}
(ROOT/'metadata'/'validation_only_decision_rule.json').write_text(json.dumps(bias_selection,indent=2))
print('Validation-only alternative decision rule:',bias_selection)
print(bias_df[['moderate_bias','high_bias','macro_f1','balanced_accuracy','predicted_minority_fraction']].head(10).to_string(index=False))

Validation-only alternative decision rule: {'weights': [1.0, 1.0, 1.0], 'selected_on': 'validation macro-F1 only', 'note': 'Fixed multiplicative probability weights define an alternative decision rule; raw probabilities remain the calibrated probabilities for PR AUC and Brier', 'top_validation_macro_f1': 0.4363248896085911, 'unadjusted_validation_macro_f1': 0.4363248896085911}
 moderate_bias  high_bias  macro_f1  balanced_accuracy  predicted_minority_fraction
           1.0        1.0  0.436325           0.410066                     0.055726
           1.5        1.0  0.433929           0.410727                     0.057459
           2.0        1.0  0.431731           0.411436                     0.059192
           1.0        1.5  0.429484           0.408119                     0.095987
           1.5        1.5  0.425548           0.407503                     0.097720
           2.0        1.5  0.425079           0.409489                     0.099453
           1.0        2.0  0.422

In [23]:
permutation_rows=[]
for replicate in [0,1,2]:
    rng=np.random.default_rng(SEED+610+replicate)
    shuffled_labels=rng.permutation(fit_block[TARGET].to_numpy())
    control=train_pipeline(fit_block,shuffled_labels,BEST_NAME,BEST_PARAMS,BEST_BALANCE,SEED+610+replicate)
    predictions=pipeline_proba(control,val_block)
    permutation_rows.append({'replicate':replicate,**numeric_summary(val_y,predictions)})
permutation_df=pd.DataFrame(permutation_rows)
permutation_df.to_csv(ROOT/'tables'/'label_permutation_negative_control_validation.csv',index=False)
print('Label-permutation negative control:')
print(permutation_df[['replicate','macro_f1','macro_ovr_ap','balanced_accuracy']].to_string(index=False))

domain_rows=[]
if 'age' in resolved and resolved['age'] in fit_block.columns:
    field=resolved['age']
    upper_bound=float(np.nanquantile(fit_block[field],0.75))
    source=fit_block.loc[fit_block[field]<upper_bound]
    distant=val_block.loc[val_block[field]>=upper_bound]
    if len(source)>500 and len(distant)>100 and source[TARGET].nunique()==3 and distant[TARGET].nunique()==3:
        heldout_fit=train_pipeline(source,source[TARGET].to_numpy(),BEST_NAME,BEST_PARAMS,BEST_BALANCE,SEED)
        q=pipeline_proba(heldout_fit,distant)
        domain_rows.append({'setting':'train_younger_evaluate_older','train_n':len(source),'eval_n':len(distant),'age_boundary':upper_bound,**numeric_summary(distant[TARGET],q)})
    else:
        print('Age-domain stress test not feasible with the available label support')
pd.DataFrame(domain_rows).to_csv(ROOT/'tables'/'synthetic_age_domain_holdout_validation.csv',index=False)
print('Age-domain shift is an INTERNAL simulated covariate transfer stress test, not an external dataset')

Label-permutation negative control:
 replicate  macro_f1  macro_ovr_ap  balanced_accuracy
         0  0.335160      0.330405           0.335227
         1  0.328662      0.329818           0.330735
         2  0.323137      0.329668           0.325980
Age-domain shift is an INTERNAL simulated covariate transfer stress test, not an external dataset


In [24]:
if CFG['run_class_specific_shap']:
    rng=np.random.default_rng(SEED)
    selection_ids=rng.choice(len(val_block),size=min(CFG['shap_sample_size'],len(val_block)),replace=False)
    shap_frame=val_block.iloc[selection_ids]
    shx=engineer(shap_frame,selected_bundle['feature_mode']).drop(columns=list(selected_bundle['excluded_features']),errors='ignore')
    shap_x=transform_preprocess(selected_bundle['preprocessor'],shx)
    shap_status={'status':'attempted','model':BEST_NAME,'sample_n':len(shap_x)}
    try:
        explainer=shap.TreeExplainer(selected_bundle['estimator'])
        shap_values=explainer.shap_values(shap_x,check_additivity=False)
        if isinstance(shap_values,list):
            shap_cube=np.stack(shap_values,axis=-1)
        else:
            shap_cube=np.asarray(shap_values)
        if shap_cube.ndim==2:
            shap_cube=shap_cube[:,:,None]
        if shap_cube.ndim==3 and shap_cube.shape[1]==len(classes) and shap_cube.shape[2]==len(shx.columns):
            shap_cube=shap_cube.transpose(0,2,1)
        if shap_cube.shape[1]!=len(shx.columns):
            raise RuntimeError('Unrecognized SHAP array shape '+str(shap_cube.shape))
        if shap_cube.shape[2]!=len(classes):
            raise RuntimeError('Expected class-wise SHAP but received '+str(shap_cube.shape))
        shap_table=[]
        for j,class_name_i in enumerate(classes):
            imp=np.abs(shap_cube[:,:,j]).mean(axis=0)
            sort=np.argsort(imp)[-min(20,len(imp)):]
            fig,ax=plt.subplots(figsize=(9,6))
            ax.barh(np.array(shx.columns)[sort],imp[sort]);ax.set_xlabel('Mean absolute SHAP (model-output scale)')
            ax.set_title('Class-specific SHAP: '+class_name_i)
            fig.tight_layout();fig.savefig(ROOT/'figures'/f'shap_class_{class_name_i}.png',dpi=CFG['figure_dpi']);plt.close(fig)
            for col,score in zip(shx.columns,imp):shap_table.append({'class':class_name_i,'feature':col,'mean_absolute_shap':score})
        pd.DataFrame(shap_table).to_csv(ROOT/'tables'/'class_specific_shap.csv',index=False)
        shap_status['status']='success'
    except Exception as error:
        shap_status.update({'status':'error','details':str(error)})
        ximp=engineer(val_block.iloc[:min(800,len(val_block))],selected_bundle['feature_mode'])
        print('SHAP failed:',error,'; class-specific SHAP is required before claiming reviewer 1 comment 7 fully addressed')
    (ROOT/'audit'/'shap_status.json').write_text(json.dumps(shap_status,indent=2))
    print(shap_status)

{'status': 'success', 'model': 'RandomForest', 'sample_n': 600}


In [25]:
def uncertainty_curves(y,p,prefix):
    conf=np.max(p,axis=1)
    errors=(p.argmax(axis=1)!=y).astype(int)
    ix=np.argsort(-conf)
    coverage=np.arange(1,len(y)+1)/len(y)
    risk=np.cumsum(errors[ix])/np.arange(1,len(y)+1)
    auc=float(np.trapz(risk,coverage)) if hasattr(np,'trapz') else float(np.trapezoid(risk,coverage))
    pd.DataFrame({'coverage':coverage,'selective_error_rate':risk}).to_csv(ROOT/'tables'/f'{prefix}_risk_coverage.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,4));ax.plot(coverage,risk)
    ax.set_xlabel('Accepted fraction (highest confidence first)');ax.set_ylabel('Classification error rate')
    fig.tight_layout();fig.savefig(ROOT/'figures'/f'{prefix}_risk_coverage.png',dpi=CFG['figure_dpi']);plt.close(fig)
    return auc

def conformal_threshold(cal_p,cal_y,alpha=0.1):
    score=1-np.clip(cal_p[np.arange(len(cal_y)),np.asarray(cal_y,dtype=int)],0,1)
    level=min(1.0,math.ceil((len(score)+1)*(1-alpha))/len(score))
    return float(np.quantile(score,level,method='higher'))
cal_raw=pipeline_proba(selected_bundle,cal_block)
cal_p=temperature_apply(cal_raw,TEMPERATURE) if CAL_METHOD=='temperature' else cal_raw
conformal_q=conformal_threshold(cal_p,cal_block[TARGET].to_numpy(),alpha=0.1)
val_sets=(1-val_proba)<=conformal_q
cv_coverage=np.mean(val_sets[np.arange(len(val_y)),val_y])
conformal_summary={'nominal_coverage':0.9,'calibration_quantile':conformal_q,'validation_marginal_coverage':float(cv_coverage),'validation_mean_set_size':float(val_sets.sum(axis=1).mean()),'interpretation':'Split conformal assumes exchangeability. No coverage guarantee under distribution shift or for individual classes.'}
conformal_summary['validation_risk_coverage_auc']=uncertainty_curves(val_y,val_proba,'validation')
(ROOT/'metadata'/'uncertainty_and_conformal_validation.json').write_text(json.dumps(conformal_summary,indent=2))
print(conformal_summary)

/tmp/ipykernel_47/2119834611.py:7: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc=float(np.trapz(risk,coverage)) if hasattr(np,'trapz') else float(np.trapezoid(risk,coverage))


{'nominal_coverage': 0.9, 'calibration_quantile': 0.5107648158152602, 'validation_marginal_coverage': 0.9034795360618584, 'validation_mean_set_size': 1.0007998933475537, 'interpretation': 'Split conformal assumes exchangeability. No coverage guarantee under distribution shift or for individual classes.', 'validation_risk_coverage_auc': 0.07832282374409766}


In [26]:
validation_predictions=val_proba.argmax(axis=1)
validation_errors=val_block[['source_row_id',TARGET]].copy()
validation_errors['prediction']=validation_predictions
validation_errors['confidence']=val_proba.max(axis=1)
validation_errors['is_error']=(validation_predictions!=val_y)
for j,n in enumerate(classes):validation_errors['prob_'+n]=val_proba[:,j]
validation_errors.to_csv(ROOT/'tables'/'validation_error_analysis.csv',index=False)
for j,c in enumerate(classes):
    pos=(val_y==j)
    if pos.sum()<2:continue
    prec,rec,thr=precision_recall_curve(pos.astype(int),val_proba[:,j])
    fig,ax=plt.subplots(figsize=(6,5));ax.plot(rec,prec,label=f'AP={average_precision_score(pos,val_proba[:,j]):.3f}')
    ax.axhline(pos.mean(),ls='--',alpha=.6,label='Prevalence baseline')
    ax.set(xlabel='Recall',ylabel='Precision',title='Validation PR curve: '+c);ax.legend();fig.tight_layout()
    fig.savefig(ROOT/'figures'/f'validation_pr_{c}.png',dpi=CFG['figure_dpi']);plt.close(fig)
print('Validation error analysis and per-class PR curves saved')

Validation error analysis and per-class PR curves saved


In [27]:
def reliability_plot(y,p,name):
    conf=p.max(axis=1); corr=(p.argmax(axis=1)==y).astype(float)
    bins=np.linspace(0,1,11)
    rows=[]
    for i,(lo,hi) in enumerate(zip(bins[:-1],bins[1:])):
        mask=(conf>=lo)&(conf<(hi if hi<1 else 1.0001))
        if mask.sum():rows.append({'bin':i,'n':int(mask.sum()),'mean_confidence':float(conf[mask].mean()),'observed_accuracy':float(corr[mask].mean())})
    tab=pd.DataFrame(rows)
    tab.to_csv(ROOT/'tables'/f'{name}_reliability_bins.csv',index=False)
    fig,ax=plt.subplots(figsize=(6,6));ax.plot([0,1],[0,1],ls='--',label='Perfect calibration');ax.plot(tab.mean_confidence,tab.observed_accuracy,marker='o',label='Observed')
    ax.set(xlabel='Confidence',ylabel='Empirical accuracy',xlim=(0,1),ylim=(0,1),title='Reliability: '+name);ax.legend();fig.tight_layout();fig.savefig(ROOT/'figures'/f'{name}_reliability.png',dpi=CFG['figure_dpi']);plt.close(fig)

reliability_plot(val_y,val_raw,'validation_before_calibration')
reliability_plot(val_y,val_proba,'validation_selected_calibration')
print('Calibration reliability diagrams saved')

Calibration reliability diagrams saved


In [28]:
def bootstrap_stratified(y,p,repetitions,seed):
    rng=np.random.default_rng(seed)
    by_class=[np.flatnonzero(np.asarray(y)==j) for j in range(len(classes))]
    results=[]
    for b in range(repetitions):
        chosen=np.concatenate([rng.choice(ids,size=len(ids),replace=True) for ids in by_class if len(ids)])
        rng.shuffle(chosen)
        results.append(numeric_summary(np.asarray(y)[chosen],np.asarray(p)[chosen]))
    table=pd.DataFrame(results)
    intervals=pd.DataFrame([{'metric':c,'mean':table[c].mean(),'lower_95':table[c].quantile(.025),'upper_95':table[c].quantile(.975)} for c in table.columns if pd.api.types.is_numeric_dtype(table[c])])
    return table,intervals

validation_boot,validation_ci=bootstrap_stratified(val_y,val_proba,CFG['bootstrap_repetitions'],seed=SEED)
validation_ci.to_csv(ROOT/'tables'/'validation_95ci_stratified_bootstrap.csv',index=False)
print(validation_ci[validation_ci.metric.isin(['macro_f1','macro_ovr_ap','balanced_accuracy','multiclass_brier',f'{"High" if "High" in classes else classes[-1]}_recall'])].to_string(index=False))

           metric     mean  lower_95  upper_95
balanced_accuracy 0.410314  0.390914  0.429427
         macro_f1 0.436582  0.413719  0.458482
 multiclass_brier 0.187968  0.182249  0.193765
     macro_ovr_ap 0.399213  0.380754  0.418003
      High_recall 0.059406  0.029703  0.094059


In [29]:
for model in [BEST_NAME]:
    if model not in ['XGBoost','LightGBM']:
        continue
    b=selected_bundle
    e=clone(b['estimator'])
    tx=transform_preprocess(b['preprocessor'],engineer(fit_block,b['feature_mode']))
    vx=transform_preprocess(b['preprocessor'],engineer(val_block,b['feature_mode']))
    if model=='XGBoost':
        e.fit(tx,fit_block[TARGET].to_numpy(),eval_set=[(tx,fit_block[TARGET].to_numpy()),(vx,val_y)],verbose=False)
        history=e.evals_result()
        tr=history['validation_0']['mlogloss']; va=history['validation_1']['mlogloss']
    else:
        e.fit(tx,fit_block[TARGET].to_numpy(),eval_set=[(tx,fit_block[TARGET].to_numpy()),(vx,val_y)],eval_metric='multi_logloss')
        history=e.evals_result_
        tr=history['training']['multi_logloss']; va=history['valid_1']['multi_logloss']
    df=pd.DataFrame({'boosting_iteration':np.arange(1,len(tr)+1),'training_logloss':tr,'validation_logloss':va})
    df.to_csv(ROOT/'tables'/'train_validation_learning_curves.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,4));ax.plot(df.boosting_iteration,df.training_logloss,label='Training');ax.plot(df.boosting_iteration,df.validation_logloss,label='Validation')
    ax.set(xlabel='Boosting iteration',ylabel='Multiclass log-loss',title='Training vs validation (test not used)');ax.legend();fig.tight_layout();fig.savefig(ROOT/'figures'/'train_validation_boosting_logloss.png',dpi=CFG['figure_dpi']);plt.close(fig)
    print('Saved legitimate training and validation iteration curves; no test-based epoch selection')

In [30]:
final_runs=[]
final_models=[]
final_test_y=test_block[TARGET].to_numpy(dtype=int)
for seed in CFG['final_seeds']:
    trained=train_pipeline(fit_block,fit_block[TARGET].to_numpy(),BEST_NAME,BEST_PARAMS,BEST_BALANCE,seed=seed)
    final_models.append(trained)
    raw_pred=pipeline_proba(trained,test_block)
    q=temperature_apply(raw_pred,TEMPERATURE) if CAL_METHOD=='temperature' else raw_pred
    scores=numeric_summary(final_test_y,q)
    final_runs.append({'seed':seed,**scores})
    joblib.dump(trained,ROOT/'weights'/f'locked_model_{BEST_NAME}_seed_{seed}.joblib',compress=3)
    pd.DataFrame(q,columns=[f'prob_{c}' for c in classes]).assign(source_row_id=test_block.source_row_id.values,truth=final_test_y).to_csv(ROOT/'probabilities'/f'locked_test_seed_{seed}.csv',index=False)
seed_df=pd.DataFrame(final_runs)
seed_df.to_csv(ROOT/'tables'/'locked_test_five_seed_metrics.csv',index=False)
print(seed_df[['seed','accuracy','balanced_accuracy','macro_f1','macro_ovr_ap','multiclass_brier']].to_string(index=False))
print('These five seeds and the fixed protocol were declared before the test was read; no selection from test results')

 seed  accuracy  balanced_accuracy  macro_f1  macro_ovr_ap  multiclass_brier
   11  0.905333           0.411185  0.442724      0.414226          0.179436
   23  0.904000           0.407879  0.440454      0.413677          0.180591
   42  0.902267           0.412071  0.442802      0.413214          0.187892
   71  0.909067           0.409679  0.442246      0.413168          0.179223
  101  0.905467           0.415668  0.447415      0.412205          0.182136
These five seeds and the fixed protocol were declared before the test was read; no selection from test results


In [31]:
all_test_prob=[]
for seed in CFG['final_seeds']:
    table=pd.read_csv(ROOT/'probabilities'/f'locked_test_seed_{seed}.csv')
    all_test_prob.append(table[[f'prob_{c}' for c in classes]].to_numpy())
test_mean=np.mean(np.stack(all_test_prob),axis=0)
test_metrics=numeric_summary(final_test_y,test_mean)
majority=int(np.bincount(fit_block[TARGET].to_numpy()).argmax())
majority_prob=np.zeros_like(test_mean)
majority_prob[:,majority]=1
majority_scores=numeric_summary(final_test_y,majority_prob)
final_table=pd.DataFrame([{'method':'Majority baseline',**majority_scores},{'method':'Five-seed mean probability (predefined)',**test_metrics}])
final_table.to_csv(ROOT/'tables'/'locked_test_final_comparison.csv',index=False)
print(final_table[['method','accuracy','macro_f1','macro_ovr_ap','balanced_accuracy','mcc','multiclass_brier']].to_string(index=False))
lock_summary={'selected_model':BEST_NAME,'selected_balance':BEST_BALANCE,'calibration':CAL_METHOD,'temperature':TEMPERATURE,'seed_list':CFG['final_seeds'],'test_evaluated_once_under_fixed_settings':True,'test_rows':len(test_block),'caution':'The original manuscript test was seen before this notebook, so this deterministic resplit should not be claimed historically untouched without original split identifiers.'}
(ROOT/'metadata'/'locked_test_protocol.json').write_text(json.dumps(lock_summary,indent=2))

                                 method  accuracy  macro_f1  macro_ovr_ap  balanced_accuracy     mcc  multiclass_brier
                      Majority baseline  0.938267  0.322717      0.333333           0.333333 0.00000          0.123467
Five-seed mean probability (predefined)  0.908667  0.443190      0.413626           0.411140 0.15298          0.178601


469

In [32]:
alternative_test_predictions=(test_mean*DECISION_WEIGHTS[None,:]).argmax(axis=1)
base_test_predictions=test_mean.argmax(axis=1)
def decision_metrics(y,hat):
    p,r,f,s=precision_recall_fscore_support(y,hat,labels=np.arange(len(classes)),zero_division=0)
    return {'accuracy':float(accuracy_score(y,hat)),'macro_f1':float(f.mean()),'balanced_accuracy':float(r.mean()),'mcc':float(matthews_corrcoef(y,hat)),**{f'{classes[j]}_precision':float(p[j]) for j in range(len(classes))},**{f'{classes[j]}_recall':float(r[j]) for j in range(len(classes))},**{f'{classes[j]}_f1':float(f[j]) for j in range(len(classes))}}
alternative_decision_test=pd.DataFrame([{'decision_rule':'calibrated_argmax',**decision_metrics(final_test_y,base_test_predictions)},{'decision_rule':'validation_tuned_cost_sensitive',**decision_metrics(final_test_y,alternative_test_predictions)}])
alternative_decision_test.to_csv(ROOT/'tables'/'locked_test_alternative_decision_rules.csv',index=False)
print('Fixed alternative decision rule evaluated on the locked test:')
print(alternative_decision_test.to_string(index=False))

reference_model=final_models[0]
speed_frame=test_block.iloc[:min(3000,len(test_block))]
start=time.perf_counter()
for _ in range(3):
    _=pipeline_proba(reference_model,speed_frame)
throughput=len(speed_frame)*3/max(time.perf_counter()-start,1e-9)
profile={'sample_n':len(speed_frame),'repeat_predictions':3,'mean_throughput_records_per_second':float(throughput),'mean_latency_milliseconds_per_record':float(1000/throughput),'hardware':platform.platform(),'note':'End-to-end batch inference including feature computation and fitted scaling, not per-person wearable device latency'}
(ROOT/'tables'/'inference_efficiency_profile.json').write_text(json.dumps(profile,indent=2))
print('Batch inference efficiency:',profile)

Fixed alternative decision rule evaluated on the locked test:
                  decision_rule  accuracy  macro_f1  balanced_accuracy     mcc  Low_precision  Moderate_precision  High_precision  Low_recall  Moderate_recall  High_recall   Low_f1  Moderate_f1  High_f1
              calibrated_argmax  0.908667   0.44319            0.41114 0.15298       0.947198            0.621359        0.021739    0.958505         0.245211     0.029703 0.952818     0.351648 0.025105
validation_tuned_cost_sensitive  0.908667   0.44319            0.41114 0.15298       0.947198            0.621359        0.021739    0.958505         0.245211     0.029703 0.952818     0.351648 0.025105
Batch inference efficiency: {'sample_n': 3000, 'repeat_predictions': 3, 'mean_throughput_records_per_second': 24392.88196947598, 'mean_latency_milliseconds_per_record': 0.040995565888907654, 'hardware': 'Linux-6.18.55+-x86_64-with-glibc2.39', 'note': 'End-to-end batch inference including feature computation and fitted scaling, 

In [33]:
cm=confusion_matrix(final_test_y,test_mean.argmax(axis=1),labels=np.arange(len(classes)))
pd.DataFrame(cm,index=['Actual '+s for s in classes],columns=['Predicted '+s for s in classes]).to_csv(ROOT/'tables'/'locked_test_confusion_matrix.csv')
fig,ax=plt.subplots(figsize=(6.5,5.5));im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
ax.set_xticks(range(len(classes)),classes);ax.set_yticks(range(len(classes)),classes)
ax.set(xlabel='Predicted class',ylabel='True class',title='Locked test confusion matrix')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
fig.tight_layout();fig.savefig(ROOT/'figures'/'locked_test_confusion_matrix.png',dpi=CFG['figure_dpi']);plt.close(fig)
for j,c in enumerate(classes):
    yc=(final_test_y==j).astype(int)
    pr,re,_=precision_recall_curve(yc,test_mean[:,j])
    fpr,tpr,_=roc_curve(yc,test_mean[:,j])
    fig,ax=plt.subplots(figsize=(6,5));ax.plot(re,pr,label=f'AP = {average_precision_score(yc,test_mean[:,j]):.3f}')
    ax.axhline(yc.mean(),ls='--',label='Prevalence');ax.set(xlabel='Recall',ylabel='Precision',title='Locked test PR: '+c);ax.legend();fig.tight_layout();fig.savefig(ROOT/'figures'/f'locked_test_PR_{c}.png',dpi=CFG['figure_dpi']);plt.close(fig)
    fig,ax=plt.subplots(figsize=(6,5));ax.plot(fpr,tpr,label=f'ROC AUC = {roc_auc_score(yc,test_mean[:,j]):.3f}')
    ax.plot([0,1],[0,1],'--');ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Locked test ROC: '+c);ax.legend();fig.tight_layout();fig.savefig(ROOT/'figures'/f'locked_test_ROC_{c}.png',dpi=CFG['figure_dpi']);plt.close(fig)
print(classification_report(final_test_y,test_mean.argmax(axis=1),target_names=classes,digits=4,zero_division=0))
reliability_plot(final_test_y,test_mean,'locked_test')

              precision    recall  f1-score   support

         Low     0.9472    0.9585    0.9528      7037
    Moderate     0.6214    0.2452    0.3516       261
        High     0.0217    0.0297    0.0251       202

    accuracy                         0.9087      7500
   macro avg     0.5301    0.4111    0.4432      7500
weighted avg     0.9109    0.9087    0.9069      7500



In [34]:
locked_reps,locked_ci=bootstrap_stratified(final_test_y,test_mean,CFG['bootstrap_repetitions'],SEED+900)
locked_ci.to_csv(ROOT/'tables'/'locked_test_95ci_stratified_bootstrap.csv',index=False)
print(locked_ci[locked_ci.metric.isin(['accuracy','macro_f1','macro_ovr_ap','balanced_accuracy','multiclass_brier',f'{"High" if "High" in classes else classes[-1]}_recall'])].to_string(index=False))

rng=np.random.default_rng(333)
paired_delta=[]
for _ in range(CFG['bootstrap_repetitions']):
    indices=np.concatenate([rng.choice(np.flatnonzero(final_test_y==k),size=np.sum(final_test_y==k),replace=True) for k in range(len(classes))])
    truth=final_test_y[indices]
    final_f1=f1_score(truth,test_mean[indices].argmax(axis=1),average='macro',zero_division=0)
    baseline_f1=f1_score(truth,np.full(len(indices),majority),average='macro',zero_division=0)
    paired_delta.append(final_f1-baseline_f1)
comparison={'point_macro_f1_improvement':test_metrics['macro_f1']-majority_scores['macro_f1'],'bootstrap_delta_macro_f1_95ci':list(np.quantile(paired_delta,[.025,.975])),'bootstrap_proportion_nonpositive':float(np.mean(np.asarray(paired_delta)<=0)),'testing_note':'Paired bootstrap comparison under a fixed test sample; exploratory significance, not proof of clinical importance'}
(ROOT/'tables'/'paired_comparison_majority_baseline.json').write_text(json.dumps(comparison,indent=2,default=float))
print(comparison)

           metric     mean  lower_95  upper_95
         accuracy 0.908631  0.904267  0.913337
balanced_accuracy 0.410757  0.391552  0.430764
         macro_f1 0.442626  0.419341  0.465697
 multiclass_brier 0.178526  0.172323  0.183610
     macro_ovr_ap 0.413713  0.394210  0.433738
      High_recall 0.029530  0.009901  0.054455
{'point_macro_f1_improvement': 0.12047363716454307, 'bootstrap_delta_macro_f1_95ci': [np.float64(0.09767126283517302), np.float64(0.14318994489888442)], 'bootstrap_proportion_nonpositive': 0.0, 'testing_note': 'Paired bootstrap comparison under a fixed test sample; exploratory significance, not proof of clinical importance'}


In [35]:
test_conformal_sets=(1-test_mean)<=conformal_q
conformal_test={'n':int(len(final_test_y)),'nominal_marginal_coverage':0.9,'observed_marginal_coverage':float(test_conformal_sets[np.arange(len(final_test_y)),final_test_y].mean()),'average_prediction_set_size':float(test_conformal_sets.sum(axis=1).mean()),'empty_set_rate':float((test_conformal_sets.sum(axis=1)==0).mean())}
for j,c in enumerate(classes):
    mask=(final_test_y==j)
    conformal_test[f'{c}_coverage']=float(test_conformal_sets[mask,j].mean()) if mask.any() else None
conformal_test['risk_coverage_auc']=uncertainty_curves(final_test_y,test_mean,'locked_test')
(ROOT/'tables'/'locked_test_conformal_and_uncertainty.json').write_text(json.dumps(conformal_test,indent=2))
print(conformal_test)

/tmp/ipykernel_47/2119834611.py:7: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc=float(np.trapz(risk,coverage)) if hasattr(np,'trapz') else float(np.trapezoid(risk,coverage))


{'n': 7500, 'nominal_marginal_coverage': 0.9, 'observed_marginal_coverage': 0.9089333333333334, 'average_prediction_set_size': 1.0010666666666668, 'empty_set_rate': 0.0005333333333333334, 'Low_coverage': 0.958789256785562, 'Moderate_coverage': 0.24521072796934865, 'High_coverage': 0.0297029702970297, 'risk_coverage_auc': 0.07291540825063761}


In [36]:
versions={}
for name in ['numpy','pandas','sklearn','scipy','imbalanced_learn','xgboost','lightgbm','shap','kagglehub']:
    try:
        import importlib
        obj=importlib.import_module(name if name!='imbalanced_learn' else 'imblearn')
        versions[name]=str(getattr(obj,'__version__','unknown'))
    except Exception as err:
        versions[name]=str(err)
metadata={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'platform':platform.platform(),'package_versions':versions,'configuration':CFG,'class_mapping':lookup,'source_features':feature_candidates,'class_names':classes,'label_rule_status':label_rule_status,'full_original_label_generator_verified':False,'external_validation_performed':False,'grouping_variable_used':GROUP_COL,'grouped_patient_validation_performed':bool(GROUP_COL),'data_source_is_synthetic':True,'artificial_shift_analyses_are_external_validation':False,'original_7_engineered_features_exactly_reproduced':False,'note':'Four manuscript formulae reproduced if source columns exist; remaining three are explicit new replacements and should be reported as such. Original label generator still requires authoritative evidence.'}
(ROOT/'metadata'/'full_reproducibility_manifest.json').write_text(json.dumps(metadata,indent=2,default=str))
revision_map={
 'R1-01_target':{'files':['audit/potential_label_generation_source_lines.csv','audit/target_rule_recoverability.json'],'status':'blocked until original target formula and thresholds confirmed from creator/source code'},
 'R1-02_circularity':{'files':['tables/feature_ablation_validation.csv','audit/target_rule_recoverability.json'],'status':'diagnostic complete; causal/source-rule circularity remains unproven without generator'},
 'R1-03_terminology':{'files':['metadata/full_reproducibility_manifest.json'],'status':'synthetic outcome must be described without clinical equivalence'},
 'R1-04_11_validation_hyperparameters':{'files':['tables/fivefold_comprehensive_cv.csv','tables/validation_model_comparison.csv','tables/conditional_nested_five_by_three_cv.csv'],'status':'analysis provided'},
 'R1-05_imbalance':{'files':['tables/fivefold_cv_ranked.csv'],'status':'four strategies evaluated'},
 'R1-06_rare_class_ci':{'files':['tables/locked_test_95ci_stratified_bootstrap.csv','tables/locked_test_final_comparison.csv','tables/locked_test_five_seed_metrics.csv'],'status':'analysis provided'},
 'R1-07_class_specific_shap':{'files':['audit/shap_status.json','tables/class_specific_shap.csv'],'status':'check shap status'},
 'R1-08_sample_size':{'files':['tables/sample_size_robustness_validation.csv'],'status':'evaluation of development-pool sizes; validate full-fit availability'},
 'R1-09_10_feature_definitions':{'files':['metadata/feature_definitions.json'],'status':'all revised engineering formulas transparent; unknown original three not misrepresented'},
 'R1-12_prose':{'files':[],'status':'manual manuscript edit required'},
 'R1-13_references':{'files':[],'status':'manual reference cross-check required'},
 'R1-14_reproducibility':{'files':['metadata/dataset_manifest.json','metadata/fit_source_identifiers.csv','metadata/configuration.json'],'status':'code and identifiers included'},
 'R2_grammar':{'files':[],'status':'manual manuscript editing required'}
}
(ROOT/'metadata'/'reviewer_requirement_map.json').write_text(json.dumps(revision_map,indent=2))
for folder in ['figures','tables','weights','logs','metadata','audit','probabilities']:
    output=ROOT.parent/f'{folder}_metabolic_revision.zip'
    with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for file in sorted((ROOT/folder).rglob('*')):
            if file.is_file():z.write(file,arcname=f'{folder}/{file.relative_to(ROOT/folder)}')
    print('ZIP:',output,'MB:',round(output.stat().st_size/(1024*1024),2))
print('ALL ANALYSES COMPLETE. Check audit warnings; never infer clinical validity or guaranteed citation impact.')

ZIP: /kaggle/working/figures_metabolic_revision.zip MB: 2.91
ZIP: /kaggle/working/tables_metabolic_revision.zip MB: 0.51
ZIP: /kaggle/working/weights_metabolic_revision.zip MB: 47.3
ZIP: /kaggle/working/logs_metabolic_revision.zip MB: 0.0
ZIP: /kaggle/working/metadata_metabolic_revision.zip MB: 0.19
ZIP: /kaggle/working/audit_metabolic_revision.zip MB: 0.0
ZIP: /kaggle/working/probabilities_metabolic_revision.zip MB: 1.8
ALL ANALYSES COMPLETE. Check audit warnings; never infer clinical validity or guaranteed citation impact.
